# Stage 03: Feature Engineering & Time Dynamics

## Overview
This notebook constructs the feature set for client **MT_196**:
- **Cyclical Time Encoding**: Sine and cosine transformations of time-of-day and day-of-week
- **Dynamic 24-Hour Baseline**: Rolling mean and rolling standard deviation over a 96-step window (24 hours)
- **Local Deviation & Rolling Z-Score**: Quantifying departure from the client's recent baseline
- **Lag Features**: Immediate previous consumption reading (`previous_reading_kw`)
- Saving the final feature dataset to `data/processed/MT_196_features.csv` and visual artifacts to `outputs/figures/`

In [ ]:
import csv
from collections import deque
from datetime import datetime
from math import cos, pi, sin, sqrt
from pathlib import Path
import matplotlib.pyplot as plt

# Paths
root = Path('..') if Path('..').resolve().name == 'ML-MINI-PROJECT' else Path('.')
source = root / 'data' / 'processed' / 'LD2011_2014_clean.csv'
dest = root / 'data' / 'processed' / 'MT_196_features.csv'
fig_dir = root / 'outputs' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

print(f"Source clean data exists: {source.exists()}")

## 1. Feature Formulations
### A. Continuous Cyclical Encoding
Because 23:45 is only 15 minutes away from 00:00, linear hour encoding creates an artificial numerical jump. We map time into continuous circular coordinates:
$$\text{time\_angle} = \frac{2\pi \cdot (\text{hour} \times 60 + \text{minute})}{1440}, \quad \sin(\text{time\_angle}), \cos(\text{time\_angle})$$
$$\text{week\_angle} = \frac{2\pi \cdot \text{weekday}}{7}, \quad \sin(\text{week\_angle}), \cos(\text{week\_angle})$$

### B. Dynamic 24-Hour Rolling Baseline (96 Readings)
$$\mu_{24h} = \frac{1}{96} \sum_{i=1}^{96} x_{t-i}, \quad \sigma_{24h} = \sqrt{\frac{1}{96} \sum_{i=1}^{96} (x_{t-i} - \mu_{24h})^2}$$
$$\text{deviation} = x_t - \mu_{24h}, \quad z\text{-score} = \frac{x_t - \mu_{24h}}{\sigma_{24h}}$$

In [ ]:
# Check or run feature building
if dest.exists():
    print(f"Features file already built: {dest} ({dest.stat().st_size / (1024**2):.1f} MB)")
else:
    print(f"Building features to {dest}...")
    client = "MT_196"
    window_size = 96
    window = deque()
    window_sum = 0.0
    window_sum_sq = 0.0
    rows = 0

    with source.open(encoding='utf-8', newline='') as f_in, dest.open('w', encoding='utf-8', newline='') as f_out:
        reader = csv.DictReader(f_in)
        writer = csv.writer(f_out)
        writer.writerow([
            "timestamp", "consumption_kw", "hour", "day_of_week",
            "is_weekend", "previous_reading_kw", "previous_24h_mean_kw",
            "deviation_from_previous_24h_kw", "previous_24h_z_score",
        ])

        for row in reader:
            t_str = row["timestamp"]
            t = datetime.strptime(t_str, "%Y-%m-%d %H:%M:%S")
            curr = float(row[client])
            prev = window[-1] if window else ""

            if len(window) == window_size:
                baseline = window_sum / window_size
                var = max(window_sum_sq / window_size - baseline**2, 0.0)
                std_dev = sqrt(var)
                dev = curr - baseline
                z = dev / std_dev if std_dev > 0 else ""
            else:
                baseline = dev = z = ""

            writer.writerow([
                t_str, curr, t.hour, t.weekday(), int(t.weekday() >= 5),
                prev, baseline, dev, z
            ])

            if len(window) == window_size:
                oldest = window.popleft()
                window_sum -= oldest
                window_sum_sq -= oldest**2

            window.append(curr)
            window_sum += curr
            window_sum_sq += curr**2
            rows += 1

    print(f"Saved {rows:,} feature rows to {dest}")

## 2. Visualizing Rolling Baseline and Z-Score Dynamics
Let's inspect the behavior of the rolling mean and Z-score over a 10-day period.

In [ ]:
t_list, kw_list, baseline_list, z_list = [], [], [], []
with dest.open(encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if not row["previous_24h_mean_kw"]:
            continue
        t = datetime.strptime(row["timestamp"], "%Y-%m-%d %H:%M:%S")
        t_list.append(t)
        kw_list.append(float(row["consumption_kw"]))
        baseline_list.append(float(row["previous_24h_mean_kw"]))
        z_list.append(float(row["previous_24h_z_score"]))
        if len(t_list) >= 96 * 10: # 10 days
            break

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax1.plot(t_list, kw_list, label='Actual Consumption (kW)', color='#1f77b4', linewidth=1.2)
ax1.plot(t_list, baseline_list, label='Rolling 24h Mean Baseline', color='#ff7f0e', linestyle='--', linewidth=1.5)
ax1.set_ylabel('Consumption (kW)')
ax1.set_title('MT_196 Consumption vs 24-Hour Rolling Baseline', fontsize=11, fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(True, linestyle=':', alpha=0.6)

ax2.plot(t_list, z_list, label='Rolling Z-Score', color='#d62728', linewidth=1.0)
ax2.axhline(3.0, color='gray', linestyle='--', label='Threshold (|z| = 3.0)')
ax2.axhline(-3.0, color='gray', linestyle='--')
ax2.set_ylabel('Z-Score')
ax2.set_xlabel('Timestamp')
ax2.legend(loc='upper right')
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
out_chart = fig_dir / 'rolling_features_MT_196.png'
plt.savefig(out_chart, dpi=150)
plt.show()
print(f"Saved figure to {out_chart}")

## Summary of Feature Engineering
- **Cyclical Invariance**: Preserves time continuity across day and week boundaries.
- **Dynamic Context**: 24-hour baseline adapts to gradual shifts and seasonal trends.
- **Ready for Modeling**: Combined 6-dimensional feature vector `[consumption_kw, z_score, sin(time), cos(time), sin(week), cos(week)]` encapsulates both amplitude and temporal context.